# Ablation 6 -- Chip x Outlier-Detection x Model Ablation

Analyses `ablations/ablation6_chip_outlier_model_ablation.py`'s results: `cnn_gru_dual` vs
`cnn_gru_dual_attn_recon`, each with no filtering / `amf_send_5` / `lstm_ae_glb` outlier
filtering, on `ori_curve_sg_p4_norm`, across chips 01-03 of `POC_DDM_final_nc_subtract`
(the only 3 chips with a matched nc_subtract sibling -- see `abl6_chip_outlier_model_ablation.sh`).

Reuses `08_statistical_comparison.py`'s Friedman + pairwise Wilcoxon (Holm-Bonferroni
corrected) framework for significance testing, same as `ablation_significance_analysis.ipynb`
-- no test logic is reimplemented here.

In [ ]:
import os, sys, importlib.util
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Run from the notebook's own directory's parent (main/), matching house style.
try:
    _nb = globals().get('__vsc_ipynb_file__')
    if _nb:
        os.chdir(os.path.dirname(os.path.dirname(_nb)))
except Exception:
    pass

%load_ext autoreload
%autoreload 2
import config

# 08_statistical_comparison.py can't be `import`ed (starts with a digit) -- load it as
# a module directly. exec_module runs it with __name__ != '__main__', so the CLI/argparse
# block at the bottom does not execute; only the reusable functions/constants do.
_spec = importlib.util.spec_from_file_location(
    "stat_comparison", str(Path.cwd() / "08_statistical_comparison.py"))
stat_comparison = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(stat_comparison)

# stat_comparison imports matplotlib and calls matplotlib.use("Agg") at module load --
# re-assert the inline backend afterwards so figures render in this notebook.
%matplotlib inline

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.color': '#e0e0e0',
    'grid.linestyle': '--',
    'grid.alpha': 0.7,
    'font.size': 10,
})
print("CWD:", os.getcwd())

## Constants

Mirrors `ablations/ablation6_chip_outlier_model_ablation.py`'s `DATASETS`/`MODELS`/`OUTLIER_FILTERS`
exactly -- keep these in sync if the script's lists change.

In [ ]:
from IPython.display import HTML, display

EXP_FOLDER = f"{config.FINAL_EXP_FOLDER}_nc_subtract"
CURVE_TYPE = "ori_curve_sg_p4_norm"

# Mirrors ablations/ablation6_chip_outlier_model_ablation.py's DATASETS exactly --
# keep in sync if the script's list changes. Chip 04 only has usable nc_subtract
# results once 01_curve_preprocessing_v6.py --nc_subtract has been run for it (see
# slurm_jobs/abl6_chip_outlier_model_ablation.sh, which now generates it on the fly).
CHIP_NAMES = [
    "D20260806_E00_C00_F4500KHz_U_DDM_01_06",
    "D20260807_E00_C00_F4500KHz_U_DDM_02_07",
    "D20260808_E00_C00_F4500KHz_U_DDM_03_01",
    "D20260810_E00_C00_F4500KHz_U_DDM_04_01",
]
CHIP_LABELS = {
    "D20260806_E00_C00_F4500KHz_U_DDM_01_06": "Chip 01",
    "D20260807_E00_C00_F4500KHz_U_DDM_02_07": "Chip 02",
    "D20260808_E00_C00_F4500KHz_U_DDM_03_01": "Chip 03",
    "D20260810_E00_C00_F4500KHz_U_DDM_04_01": "Chip 04",
}

# gnn_gat (Keras GAT-style spatial reconstruction, see model_utils_gnn_recon.py's
# create_cnn_gru_dual_gat_recon_model) trains alongside the other two Keras models in
# the same ablation6 run and lands in the same results file -- same MODEL_KEY_MAP-driven
# loader picks it up automatically, no separate load path needed.
MODELS = ['cnn_gru_dual', 'cnn_gru_dual_attn_recon', 'gnn_gat']
MODEL_PRINT_MAP = {
    'cnn_gru_dual':            'CNN-BiGRU Dual Branch',
    'cnn_gru_dual_attn_recon': 'CNN-BiGRU Dual + Attn Recon',
    'gnn_gat':                 'GNN (GAT)',
}
MODEL_COLORS = {
    'cnn_gru_dual':            '#BB8FCE',
    'cnn_gru_dual_attn_recon': '#F7DC6F',
    'gnn_gat':                 '#5DADE2',
}

OUTLIER_FILTERS = [None, 'amf_label_amf_send_5',
                   f'lstm_ae_glb_ds{config.AE_DOWNSAMPLE_FACTOR}_label_elbow']
FILTER_PRINT_MAP = {
    None:                                                            'No Filter',
    'amf_label_amf_send_5':                                         'AMF (send_5)',
    f'lstm_ae_glb_ds{config.AE_DOWNSAMPLE_FACTOR}_label_elbow':      'LSTM-AE (global)',
}
FILTER_COLORS = {
    None:                                                            '#85C1E9',
    'amf_label_amf_send_5':                                         '#82E0AA',
    f'lstm_ae_glb_ds{config.AE_DOWNSAMPLE_FACTOR}_label_elbow':      '#EC7063',
}

ALPHA = 0.05
METRIC = 'accuracy'

## Load results

Adapts `ablation6_chip_outlier_model_ablation_performances.joblib`'s flat `{filter: {...}}`
schema (same per-model `y_preds_AC_*_`/`y_trues_` keys as `classification_performances.joblib`)
into the `{curve_type: {filter: {model: {fold_metrics, mean_metric, y_true_all, y_pred_all,
n_folds}}}}` shape `stat_comparison.build_accuracy_matrix`/`_get_value` expect. Mirrors
`ablation_significance_analysis.ipynb`'s `load_ablation_exp_data` exactly.

In [ ]:
def load_ablation_exp_data(joblib_path, models, curve_type=CURVE_TYPE, metric=METRIC):
    joblib_path = Path(joblib_path)
    if not joblib_path.exists():
        return None
    raw = joblib.load(joblib_path)

    out_filters = {}
    for filter_key, filter_results in raw.items():
        if "y_trues_" not in filter_results:
            continue
        y_trues_ = filter_results["y_trues_"]
        n_folds = len(y_trues_)
        if n_folds == 0:
            continue

        out_filters[filter_key] = {}
        for model_key in models:
            preds_key = config.MODEL_KEY_MAP[model_key][0]
            if preds_key not in filter_results:
                continue
            y_preds_ = filter_results[preds_key]
            if len(y_preds_) != n_folds:
                continue

            fold_metrics, y_true_parts, y_pred_parts = [], [], []
            for fi in range(n_folds):
                y_true = np.asarray(y_trues_[fi])
                y_pred = np.asarray(y_preds_[fi])
                if len(y_true) == 0 or len(y_true) != len(y_pred):
                    continue
                fold_metrics.append(stat_comparison._compute_metric(y_true, y_pred, metric))
                y_true_parts.append(y_true)
                y_pred_parts.append(y_pred)

            if not fold_metrics:
                continue
            out_filters[filter_key][model_key] = {
                "fold_metrics": fold_metrics,
                "mean_metric":  float(np.mean(fold_metrics)),
                "y_true_all":   np.concatenate(y_true_parts),
                "y_pred_all":   np.concatenate(y_pred_parts),
                "n_folds":      len(fold_metrics),
            }
    return {curve_type: out_filters}


def load_ablation_across_chips(exp_folder, chip_names, models, curve_type=CURVE_TYPE, metric=METRIC):
    exp_data_list = []
    for name in chip_names:
        path = (Path(exp_folder) / name / "ablations"
                / "ablation6_chip_outlier_model_ablation_performances.joblib")
        data = load_ablation_exp_data(path, models, curve_type=curve_type, metric=metric)
        if data is not None and any(data[curve_type].values()):
            exp_data_list.append((name, data))
        else:
            print(f'  [WARN] no usable results for {name} at {path}')
    return exp_data_list


exp_data = load_ablation_across_chips(EXP_FOLDER, CHIP_NAMES, MODELS)
print(f'Loaded {len(exp_data)} / {len(CHIP_NAMES)} chips.')

## Per-chip barchart

One grouped bar chart per chip: x = outlier filter, bars = model. Dashed edge = the
`None` (no filtering) baseline; ★ = best model for that filter.

In [ ]:
def plot_chip_barchart(chip_name, exp_data_for_chip, models, filters,
                       model_colors, model_print_map, filter_print_map,
                       curve_type=CURVE_TYPE):
    entry_by_filter = exp_data_for_chip[curve_type]
    present_filters = [f for f in filters if f in entry_by_filter]
    present_models = [m for m in models
                      if any(m in entry_by_filter.get(f, {}) for f in present_filters)]

    acc = np.full((len(present_filters), len(present_models)), np.nan)
    for i, f in enumerate(present_filters):
        for j, m in enumerate(present_models):
            res = entry_by_filter.get(f, {}).get(m)
            if res is not None:
                acc[i, j] = res['mean_metric'] * 100
    best_j = np.nanargmax(acc, axis=1)

    n_models = len(present_models)
    width = 0.8 / n_models
    x = np.arange(len(present_filters))

    fig, ax = plt.subplots(figsize=(2 * len(present_filters) + 2, 4))
    for j, m in enumerate(present_models):
        offsets = x + (j - (n_models - 1) / 2) * width
        bars = ax.bar(offsets, acc[:, j], width=width, color=model_colors.get(m, '#95A5A6'),
                      label=model_print_map.get(m, m), zorder=3)
        for i, bar in enumerate(bars):
            v = acc[i, j]
            if np.isnan(v):
                continue
            is_baseline = present_filters[i] is None
            is_best = j == best_j[i]
            if is_baseline:
                bar.set_linestyle('--')
                bar.set_edgecolor('#555555')
                bar.set_linewidth(1.3)
            if is_best:
                ax.text(bar.get_x() + bar.get_width() / 2, v + 2.6, '★',
                        ha='center', va='bottom', fontsize=11, color='#B7950B', zorder=4)
            ax.text(bar.get_x() + bar.get_width() / 2, v + 0.6, f'{v:.2f}',
                    ha='center', va='bottom', fontsize=8,
                    fontweight='bold' if is_best else 'normal')

    ax.set_xticks(x)
    ax.set_xticklabels([filter_print_map.get(f, str(f)) for f in present_filters])
    ax.set_ylabel('Accuracy (%)')
    ax.set_ylim(0, 112)
    fig.suptitle(f'{CHIP_LABELS.get(chip_name, chip_name)}: Accuracy by Outlier Filter and Model',
                fontsize=13, y=1.03)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=n_models, frameon=False)
    fig.tight_layout()
    return fig


for chip_name, chip_data in exp_data:
    fig = plot_chip_barchart(chip_name, chip_data, MODELS, OUTLIER_FILTERS,
                             MODEL_COLORS, MODEL_PRINT_MAP, FILTER_PRINT_MAP)
    plt.show()

## All combinations, all chips

One combined bar chart: x = chip, bars = every (outlier filter x model) combination
(6 bars per chip).

In [ ]:
def plot_combined_barchart(exp_data, models, filters, model_print_map, filter_print_map,
                          curve_type=CURVE_TYPE):
    chip_names = [n for n, _ in exp_data]
    data_by_name = dict(exp_data)

    combos = [(f, m) for f in filters for m in models]
    combo_labels = [f'{filter_print_map.get(f, str(f))} × {model_print_map.get(m, m)}'
                    for f, m in combos]
    combo_colors = config.get_palette(combo_labels)

    acc = np.full((len(chip_names), len(combos)), np.nan)
    for i, name in enumerate(chip_names):
        entry_by_filter = data_by_name[name][curve_type]
        for k, (f, m) in enumerate(combos):
            res = entry_by_filter.get(f, {}).get(m)
            if res is not None:
                acc[i, k] = res['mean_metric'] * 100
    best_k = np.nanargmax(acc, axis=1)

    n_combos = len(combos)
    width = 0.8 / n_combos
    x = np.arange(len(chip_names))

    fig, ax = plt.subplots(figsize=(2.5 * len(chip_names) + 4, 5))
    for k, label in enumerate(combo_labels):
        offsets = x + (k - (n_combos - 1) / 2) * width
        bars = ax.bar(offsets, acc[:, k], width=width, color=combo_colors[label],
                      label=label, zorder=3)
        for i, bar in enumerate(bars):
            v = acc[i, k]
            if np.isnan(v):
                continue
            is_best = k == best_k[i]
            if is_best:
                ax.text(bar.get_x() + bar.get_width() / 2, v + 2.6, '★',
                        ha='center', va='bottom', fontsize=10, color='#B7950B', zorder=4)
            ax.text(bar.get_x() + bar.get_width() / 2, v + 0.6, f'{v:.1f}',
                    ha='center', va='bottom', fontsize=6, rotation=90,
                    fontweight='bold' if is_best else 'normal')

    ax.set_xticks(x)
    ax.set_xticklabels([CHIP_LABELS.get(n, n) for n in chip_names])
    ax.set_ylabel('Accuracy (%)')
    ax.set_ylim(0, 118)
    fig.suptitle('Chip x Outlier-Filter x Model: Every Combination', fontsize=13, y=1.02)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=3, frameon=False, fontsize=8)
    fig.tight_layout()
    return fig


fig_combined = plot_combined_barchart(exp_data, MODELS, OUTLIER_FILTERS,
                                      MODEL_PRINT_MAP, FILTER_PRINT_MAP)
plt.show()

## Mean across chips

Collapses the 3 chips into one bar per combination -- which (filter, model) pair wins
on average.

In [ ]:
def plot_mean_barchart(exp_data, models, filters, model_print_map, filter_print_map,
                      curve_type=CURVE_TYPE):
    data_by_name = dict(exp_data)
    combos = [(f, m) for f in filters for m in models]
    combo_labels = [f'{filter_print_map.get(f, str(f))} × {model_print_map.get(m, m)}'
                    for f, m in combos]
    combo_colors = config.get_palette(combo_labels)

    means, stds = [], []
    for f, m in combos:
        vals = []
        for _, chip_data in exp_data:
            res = chip_data[curve_type].get(f, {}).get(m)
            if res is not None:
                vals.append(res['mean_metric'] * 100)
        means.append(np.mean(vals) if vals else np.nan)
        stds.append(np.std(vals) if len(vals) > 1 else 0.0)

    order = np.argsort(means)[::-1]
    fig, ax = plt.subplots(figsize=(2 * len(combos) + 2, 4.5))
    x = np.arange(len(combos))
    bars = ax.bar(x, [means[i] for i in order], yerr=[stds[i] for i in order], capsize=4,
                  color=[combo_colors[combo_labels[i]] for i in order], zorder=3)
    for i, bar in enumerate(bars):
        v = means[order[i]]
        if np.isnan(v):
            continue
        ax.text(bar.get_x() + bar.get_width() / 2, v + stds[order[i]] + 1.0, f'{v:.2f}',
                ha='center', va='bottom', fontsize=9,
                fontweight='bold' if i == 0 else 'normal')
    ax.set_xticks(x)
    ax.set_xticklabels([combo_labels[i] for i in order], rotation=30, ha='right')
    ax.set_ylabel('Mean Accuracy (%) across chips 01-03')
    ax.set_ylim(0, 112)
    fig.suptitle('Mean Accuracy Across Chips 01-03, Ranked', fontsize=13, y=1.02)
    fig.tight_layout()
    return fig


fig_mean = plot_mean_barchart(exp_data, MODELS, OUTLIER_FILTERS, MODEL_PRINT_MAP, FILTER_PRINT_MAP)
plt.show()

## Significance: does outlier filtering help?

Friedman test across the 3 chips (N=3, matched blocks) comparing the 3 outlier filters,
run separately per model, then pairwise Wilcoxon signed-rank (Holm-Bonferroni corrected)
against the `None` baseline. Reuses `stat_comparison.run_comparison`'s `outlier_filters`
axis untouched -- same framework as `08_statistical_comparison.py` and
`ablation_significance_analysis.ipynb`.

In [ ]:
def show_result(stats, figs, test_type, metric_name='Accuracy', alpha=ALPHA, title=None):
    if title:
        display(HTML(f'<h3>{title}</h3>'))
    if not stats or 'error' in (stats or {}):
        display(HTML(f'<p style="color:#e74c3c">{(stats or {}).get("error", "No data.")}</p>'))
        return
    html = stat_comparison.build_tab_content(stats, figs, test_type, metric_name, alpha)
    display(HTML(html))
    for fig in figs.values():
        display(fig)
        plt.close(fig)


filter_stats = {}
for model_key in MODELS:
    stats, figs, test_type, _ = stat_comparison.run_comparison(
        exp_data,
        compare_axis="outlier_filters",
        all_conditions=OUTLIER_FILTERS,
        condition_print_map=FILTER_PRINT_MAP,
        fixed={"curve_type": CURVE_TYPE, "model": model_key},
        baseline_key=None,
        metric=METRIC,
        alpha=ALPHA,
    )
    filter_stats[model_key] = (stats, figs, test_type)
    show_result(stats, figs, test_type,
               title=f'Outlier Filter Comparison -- {MODEL_PRINT_MAP.get(model_key, model_key)}')

## Significance: does the attn_recon architecture help?

Same 3 chips, comparing `cnn_gru_dual` vs `cnn_gru_dual_attn_recon`, fixed at the `None`
(no filtering) baseline.

In [ ]:
stats_model, figs_model, tt_model, _ = stat_comparison.run_comparison(
    exp_data,
    compare_axis="models",
    all_conditions=MODELS,
    condition_print_map=MODEL_PRINT_MAP,
    fixed={"curve_type": CURVE_TYPE, "filter": None},
    baseline_key="cnn_gru_dual",
    metric=METRIC,
    alpha=ALPHA,
)
show_result(stats_model, figs_model, tt_model, title="Model Comparison (No Filter)")

## Optional: save a combined HTML report

Writes the same tabbed HTML format `08_statistical_comparison.py` produces, so it can be
shared/reopened without re-running this notebook.

In [ ]:
# from html_utils import build_tabbed_html

# SAVE_REPORT = False  # flip to True to write the file

# if SAVE_REPORT:
#     tabs = [(f"filters_{m}", f"Filters -- {MODEL_PRINT_MAP.get(m, m)}",
#             stat_comparison.build_tab_content(*filter_stats[m][:3], 'Accuracy', ALPHA))
#            for m in MODELS]
#     tabs.append(("models", "Model Comparison",
#                 stat_comparison.build_tab_content(stats_model, figs_model, tt_model, 'Accuracy', ALPHA)))
#     out_dir = Path(EXP_FOLDER) / "ablations" / "stat_comparison"
#     out_dir.mkdir(parents=True, exist_ok=True)
#     out_path = out_dir / "ablation6_chip_ablation_report.html"
#     build_tabbed_html("Ablation 6: Chip Outlier/Model Ablation", tabs, out_path)
#     print(f'[SAVED] {out_path}')